# Phase 1 — prepare Qwen3.5 coder data on CPU
Use a Colab **CPU-only runtime**. This notebook prepares the complete Stage-1 and repository datasets, validates them, and uploads the reusable bundle to a private Hugging Face Dataset repository. It consumes no GPU runtime while preparing data. Do not disconnect until the final upload cell succeeds.

In [ ]:
from google.colab import userdata
import shutil, subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/SSusantAchary/coder-SFT.git'
PROJECT_ROOT = Path('/content/coder_SFT')
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'{PROJECT_ROOT} exists but is not a valid coder-SFT checkout')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
WORK_ROOT = Path('/content/qwen35-data-prep')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage('/content')
print(f'Colab disk: {disk.total / 2**30:.1f} GiB total, {disk.free / 2**30:.1f} GiB free')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PROJECT_ROOT / 'requirements-data.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-c', 'import coder_sft; print(coder_sft.__version__)'], check=True)

In [ ]:
import os, sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['CODER_SFT_WORKDIR'] = str(WORK_ROOT)
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add a write-enabled HF_TOKEN to Colab Secrets and grant this notebook access.')
os.environ['HF_TOKEN'] = hf_token
from huggingface_hub import HfApi
HF_OWNER = HfApi(token=hf_token).whoami()['name']
DATA_REPO_ID = f'{HF_OWNER}/Qwen3.5-2B-Coder-Data'
DATA_REPO_PRIVATE = True
BASE = str(PROJECT_ROOT / 'configs/base.yaml')
DATA = str(PROJECT_ROOT / 'configs/data_v1.yaml')
print(f'Prepared data will be uploaded to: {DATA_REPO_ID} (private={DATA_REPO_PRIVATE})')

## Build the reusable data bundle
This is the long CPU/network phase. It creates normalized JSONL plus provenance manifests under `/content/qwen35-data-prep/data`. Failed or oversized SWE-smith repositories are recorded in the manifest.

In [ ]:
subprocess.run(['prepare-data', '--config', BASE, '--config', DATA], check=True)
subprocess.run(['build-repo-context', '--config', BASE, '--config', DATA], check=True)

## Validate and upload
The command refuses to upload unless both datasets and both complete manifests exist. The token stays in memory; it is never written into the bundle. After this succeeds, you may delete the CPU runtime and open the GPU notebook.

In [ ]:
upload = ['sync-data', 'upload', '--config', BASE, '--repo-id', DATA_REPO_ID]
if not DATA_REPO_PRIVATE:
    upload.append('--public')
subprocess.run(upload, check=True)
print(f'Data handoff complete: https://huggingface.co/datasets/{DATA_REPO_ID}')
print('You can now release this CPU runtime and start 02_Train_and_Publish_GPU.ipynb.')